# MLflow Experiment Tracking — AI Engineering Interview Prep

Log experiments, compare runs, register models with MLflow.

In [ ]:
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# Setup
data = load_breast_cancer()
X, y = data.data, data.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

mlflow.set_tracking_uri("mlruns")  # local tracking
mlflow.set_experiment("breast_cancer_classification")
print(f"MLflow version: {mlflow.__version__}")
print("Tracking URI:", mlflow.get_tracking_uri())

## 1. Log a Single Run

In [ ]:
def log_sklearn_experiment(model, model_name, params, X_tr, y_tr, X_te, y_te, scaler=None):
    """Train model, log params/metrics/model to MLflow."""
    with mlflow.start_run(run_name=model_name):
        # Log parameters
        mlflow.log_params(params)
        mlflow.log_param("model_name", model_name)
        mlflow.log_param("n_features", X_tr.shape[1])
        mlflow.log_param("n_train", len(X_tr))

        # Apply scaling if provided
        if scaler:
            Xtr = scaler.fit_transform(X_tr)
            Xte = scaler.transform(X_te)
        else:
            Xtr, Xte = X_tr, X_te

        # Train
        model.fit(Xtr, y_tr)
        y_pred = model.predict(Xte)
        y_prob = model.predict_proba(Xte)[:, 1] if hasattr(model, 'predict_proba') else None

        # Metrics
        acc  = accuracy_score(y_te, y_pred)
        f1   = f1_score(y_te, y_pred)
        auc  = roc_auc_score(y_te, y_prob) if y_prob is not None else None
        cv_  = cross_val_score(model, Xtr, y_tr, cv=5, scoring='roc_auc').mean()

        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("f1_score", f1)
        if auc: mlflow.log_metric("roc_auc", auc)
        mlflow.log_metric("cv_roc_auc", cv_)

        # Log model
        mlflow.sklearn.log_model(model, "model")

        run_id = mlflow.active_run().info.run_id
        print(f"{model_name:25s} acc={acc:.4f} f1={f1:.4f} auc={auc:.4f if auc else 'N/A'} run={run_id[:8]}")
        return run_id

In [ ]:
# Run multiple experiments
experiments = [
    (RandomForestClassifier(n_estimators=50, random_state=42),
     "RandomForest_50",
     {"n_estimators": 50, "max_depth": "None"},
     None),

    (RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42),
     "RandomForest_200_d10",
     {"n_estimators": 200, "max_depth": 10},
     None),

    (GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42),
     "GradientBoosting",
     {"n_estimators": 100, "learning_rate": 0.1},
     None),

    (LogisticRegression(C=1.0, max_iter=200),
     "LogisticReg_C1",
     {"C": 1.0, "solver": "lbfgs"},
     StandardScaler()),

    (LogisticRegression(C=0.01, max_iter=200),
     "LogisticReg_C0.01",
     {"C": 0.01, "solver": "lbfgs"},
     StandardScaler()),
]

run_ids = []
for model, name, params, scaler in experiments:
    run_id = log_sklearn_experiment(model, name, params, X_tr, y_tr, X_te, y_te, scaler)
    run_ids.append(run_id)

## 2. Query & Compare Runs

In [ ]:
# Load all runs from the experiment
runs = mlflow.search_runs(experiment_names=["breast_cancer_classification"])
comparison = runs[['tags.mlflow.runName', 'metrics.accuracy', 'metrics.roc_auc',
                    'metrics.f1_score', 'metrics.cv_roc_auc']].copy()
comparison.columns = ['model', 'accuracy', 'roc_auc', 'f1', 'cv_auc']
comparison = comparison.sort_values('roc_auc', ascending=False)
print(comparison.to_string(index=False, float_format='{:.4f}'.format))

## 3. Load a Logged Model

In [ ]:
# Load the best run's model
best_run_id = runs.loc[runs['metrics.roc_auc'].idxmax(), 'run_id']
print(f"Best run: {best_run_id}")

loaded_model = mlflow.sklearn.load_model(f"runs:/{best_run_id}/model")
y_pred_loaded = loaded_model.predict(X_te)
print(f"Loaded model accuracy: {accuracy_score(y_te, y_pred_loaded):.4f}")

## 4. Hyperparameter Sweeps with MLflow

In [ ]:
param_grid = [
    {"n_estimators": n, "max_depth": d}
    for n in [50, 100, 200]
    for d in [5, 10, None]
]

sweep_results = []
for params in param_grid:
    with mlflow.start_run(run_name=f"RF_n{params['n_estimators']}_d{params['max_depth']}"):
        mlflow.log_params(params)
        model = RandomForestClassifier(random_state=42, **params)
        model.fit(X_tr, y_tr)
        auc = roc_auc_score(y_te, model.predict_proba(X_te)[:,1])
        mlflow.log_metric("roc_auc", auc)
        sweep_results.append({**params, "roc_auc": auc})

sweep_df = pd.DataFrame(sweep_results).sort_values("roc_auc", ascending=False)
print("Hyperparameter sweep results:")
print(sweep_df.head())

## Interview Tips

- **MLflow core concepts**: Experiments → Runs → Parameters/Metrics/Artifacts/Models.
- **Reproducibility**: log random seeds, dataset version, preprocessing params alongside model.
- **Model registry**: use `mlflow.register_model()` to promote runs to Staging/Production stages.
- **Alternatives**: Weights & Biases (wandb), Neptune, Comet — all share similar concepts.
- **Autologging**: `mlflow.sklearn.autolog()` automatically logs params and metrics without explicit calls.
- **MLflow UI**: run `mlflow ui` in terminal to see all experiments in browser at localhost:5000.